# Phases 4 and 5: Train and evaluate SimpleCNN on Kaggle

*Made with ❤️ by Calli*

**Where this fits in the pipeline:** data → model → `[train, on a real GPU]` → `[evaluate]` → predict

This notebook takes the code we wrote and tested on the laptop and runs it on the **real** dataset with a **GPU**. The code reaches Kaggle as a zip file (no GitHub needed).

---

### One-time setup (and what to repeat when the code changes)

**A. On your laptop: pack the code into a zip.** In the project folder:
```bash
git archive --format=zip --prefix=malaria-cnn/ -o malaria-cnn-code.zip HEAD
```
`git archive` packs the files of your **latest commit** (HEAD) into a zip, inside a `malaria-cnn/` folder. Because it only takes committed files, it never includes `.venv/`, the fake data or old outputs. *Commit first*, otherwise your newest changes aren't in the zip.

**B. On Kaggle: upload the zip as a private dataset.**
1. **Datasets → New Dataset**, upload `malaria-cnn-code.zip`, title it `malaria-cnn-code`, keep it **Private**, click **Create**.
2. *When the code changes later:* make a new zip (step A), open your dataset → **New Version** → upload it.

**C. Set up this notebook.**
1. **Create → New Notebook**, then **File → Import Notebook** and upload this file.
2. Right panel → **Session options → Accelerator → GPU** (T4 or P100, either is fine).
3. **Add Input** → add **both**: `cell-images-for-detecting-malaria` (by iarunava) **and** your `malaria-cnn-code` dataset.
4. Internet is **not** needed. Click **Run All**. Expect roughly 15 to 30 minutes in total.
5. **Wait until both inputs have finished attaching** (a few minutes for the images) before Run All.

## 1. Settings

Everything this notebook needs to know, in one place. You shouldn't need to change these on Kaggle.

In [ ]:
import shutil
import sys
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")      # attached datasets live here (read-only)
KAGGLE_WORKING = Path("/kaggle/working")  # our writable folder; its contents appear in the Output tab
MODEL_NAME = "simple_cnn"
REQUIRE_GPU = True  # stop early if the GPU isn't switched on; training on CPU would take many hours

## 2. Find our code

Kaggle unpacks uploaded zips into `/kaggle/input/<dataset name>/`, but the exact folder nesting can vary. So instead of guessing a path, we **search** for our `src/train.py` and take the folder above `src/` as the project folder.

We also check the malaria dataset is attached (its images sit in folders called `Parasitized` and `Uninfected`). Each Kaggle notebook has its *own* list of inputs, so it's easy to forget one.

In [ ]:
train_scripts = sorted(KAGGLE_INPUT.rglob("src/train.py"))
print("Found:", [str(path) for path in train_scripts])

if len(train_scripts) == 0:
    raise FileNotFoundError("Code not found. Did you Add Input -> your malaria-cnn-code dataset?")
if len(train_scripts) > 1:
    raise RuntimeError("Found the code more than once. Remove the extra input so we know which version to run.")

code_in_input = train_scripts[0].parent.parent  # .../malaria-cnn/src/train.py -> .../malaria-cnn
print("Code folder:", code_in_input)

# Is the malaria dataset attached too? Look for its "Parasitized" image folder.
parasitized_folders = sorted(KAGGLE_INPUT.rglob("Parasitized"))
if len(parasitized_folders) == 0:
    raise FileNotFoundError(
        "The malaria dataset isn't attached to this notebook. Add Input -> search "
        "'cell-images-for-detecting-malaria' (by iarunava) -> click +, then Run All again."
    )
print("Malaria images found in:", parasitized_folders[0].parent)

## 3. Copy the code somewhere writable

`/kaggle/input` is **read-only**, but the first run must *write* `splits/splits.csv` (our saved train/val/test split). So we copy the project into `/kaggle/working/malaria-cnn` and run it from there.

Then we "move into" that folder (`%cd`), so that `python -m src.train` finds the `src` package, and add it to Python's search path so this notebook can import our `config` too.

In [ ]:
PROJECT_DIR = KAGGLE_WORKING / "malaria-cnn"
# dirs_exist_ok=True: if we run this cell twice, overwrite instead of failing.
# A splits.csv created by an earlier run in this session is kept.
shutil.copytree(code_in_input, PROJECT_DIR, dirs_exist_ok=True)

%cd {PROJECT_DIR}
sys.path.insert(0, str(PROJECT_DIR))

from src import config
print("Outputs will be saved to:", config.OUTPUT_DIR)
print("Split file:", config.SPLITS_CSV, "(exists)" if config.SPLITS_CSV.exists() else "(will be created)")

## 4. Check the GPU and record library versions

A GPU makes training 10 to 50 times faster. If this cell stops with an error, switch the accelerator on (setup step C.2) and run again.

We also print the library versions Kaggle uses, so we can pin the same versions in `requirements.txt` on the laptop.

In [ ]:
import matplotlib
import numpy
import pandas
import PIL
import sklearn
import torch
import torchvision
import tqdm

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif REQUIRE_GPU:
    raise RuntimeError("No GPU! Session options -> Accelerator -> GPU, then Run All again.")
else:
    print("WARNING: no GPU, running on CPU (only OK for testing)")

library_versions = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "numpy": numpy.__version__,
    "pandas": pandas.__version__,
    "Pillow": PIL.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "tqdm": tqdm.__version__,
}
for library, version in library_versions.items():
    print(f"{library:>13}: {version}")

## 5. Sanity check: can the model memorise 16 real cells?

Same check as on the laptop (`--overfit-batch`), now on **real** images and the GPU. If the loss can't fall close to zero, something is wrong, and the notebook stops here instead of wasting GPU time.

**The split:** the code zip contains `splits/splits.csv` (saved after the first run), so the same split is loaded every time. Check the table it prints: **143 / 28 / 29 patients**. If the file were missing, the code would find, de-duplicate and split all images again (about a minute).

*`{sys.executable}` means "the same Python this notebook uses". `_exit_code` is set by Jupyter after a `!` command: 0 means success.*

In [ ]:
!{sys.executable} -m src.train --model {MODEL_NAME} --data-dir {KAGGLE_INPUT} --overfit-batch

if _exit_code != 0:
    raise RuntimeError("The overfit check failed (or crashed). The output above shows why.")

## 6. Train (or reuse a model from this session)

Up to 30 epochs; early stopping usually ends it sooner. One line is printed per epoch:

- **train loss / val loss:** should both go down. Val loss rising while train loss keeps falling = overfitting.
- **val acc / val recall:** share of all validation cells right / share of infected validation cells caught.
- **lr:** the learning rate; it halves when validation loss stalls.

The best epoch's model is saved as `simple_cnn_best.pt`. **If that file already exists from earlier in this same session, training is skipped**, so re-running the notebook for evaluation doesn't cost another 13 minutes. Training is reproducible, so a fresh session trains the identical model.

In [ ]:
checkpoint_path = config.OUTPUT_DIR / f"{MODEL_NAME}_best.pt"

if checkpoint_path.exists():
    print(f"Found {checkpoint_path.name} from earlier in this session: skipping training.")
    print("(To train again, delete that file or restart the session.)")
else:
    !{sys.executable} -m src.train --model {MODEL_NAME} --data-dir {KAGGLE_INPUT}
    if _exit_code != 0:
        raise RuntimeError("Training failed. The output above shows why.")

## 7. Look at the training curves

**Left:** train vs validation loss. **Right:** validation accuracy and recall (the axis is zoomed in, so check the numbers on it).

In [ ]:
import json

from IPython.display import Image, display

display(Image(filename=str(config.OUTPUT_DIR / f"{MODEL_NAME}_curves.png")))

with open(config.OUTPUT_DIR / f"{MODEL_NAME}_history.json") as history_file:
    history = json.load(history_file)
history_table = pandas.DataFrame(history).set_index("epoch").round(4)
print(history_table.to_string())

## 8. Evaluate on the test set (Phase 5)

This is the moment the 29 test patients have been kept for. What happens:

1. The trained model scores every **validation** and **test** cell (probability of infection).
2. **On validation data only**, we pick the *highest* threshold that still catches at least 98% of infected cells.
3. The **test** set is scored at the default threshold (0.5) and at that tuned threshold, side by side. Nothing is chosen from the test results: they're only measured.

Read the table it prints. The two columns show the trade-off: the tuned threshold should miss fewer infections, at the cost of more false alarms.

In [ ]:
!{sys.executable} -m src.evaluate --model {MODEL_NAME} --data-dir {KAGGLE_INPUT}

if _exit_code != 0:
    raise RuntimeError("Evaluation failed. The output above shows why.")

## 9. Look at the evaluation plots

- **Confusion matrices:** rows are the truth, columns the prediction. The bottom-left square is **missed infections**, the number we most want to shrink.
- **ROC curve:** recall against false-alarm rate for every possible threshold. The two dots are our two thresholds.
- **Misclassified cells:** the test cells the model got wrong at the tuned threshold. **Would you have got them right?**

In [ ]:
for plot_name in ["confusion_matrices", "roc_curve", "misclassified"]:
    display(Image(filename=str(config.OUTPUT_DIR / f"{MODEL_NAME}_{plot_name}.png")))

## 10. Gather everything into one zip to download

This packs `/kaggle/working/outputs/` into **`malaria-outputs.zip`**. To download it: right panel → **Output** → refresh if needed → hover over `malaria-outputs.zip` → **⋮** → **Download**. Do it while the session is still running: files in `/kaggle/working` disappear when the session ends.

In [ ]:
shutil.copy(config.SPLITS_CSV, config.OUTPUT_DIR / "splits.csv")

# make_archive(name without ".zip", format, folder to zip from, folder to put inside the zip)
zip_path = shutil.make_archive(
    str(KAGGLE_WORKING / "malaria-outputs"), "zip",
    root_dir=config.OUTPUT_DIR.parent, base_dir=config.OUTPUT_DIR.name,
)

for path in sorted(config.OUTPUT_DIR.iterdir()):
    if path.is_file():
        size_in_mb = path.stat().st_size / 1_000_000
        print(f"{path.name:<36} {size_in_mb:8.2f} MB")
print(f"\nZipped into {zip_path} ({Path(zip_path).stat().st_size / 1_000_000:.2f} MB)")

## 11. Summary

Copy **everything this cell prints** and check it for the red flags in PLAN.md Section 7.

In [ ]:
split_table = pandas.read_csv(config.SPLITS_CSV, dtype={"patient_id": str})
best_epoch = min(history, key=lambda record: record["val_loss"])  # the epoch with the lowest validation loss
with open(config.OUTPUT_DIR / f"{MODEL_NAME}_metrics.json") as metrics_file:
    metrics = json.load(metrics_file)

print("===== PHASE 4 + 5 SUMMARY =====")
print("library versions:", library_versions)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("\nsplit (images / patients / parasitized share):")
for split_name in ["train", "val", "test"]:
    rows = split_table[split_table["split"] == split_name]
    parasitized_share = (rows["label"] == config.LABEL_PARASITIZED).mean()
    print(f"  {split_name:<5} {len(rows):>6} images  {rows['patient_id'].nunique():>4} patients  "
          f"{parasitized_share:.1%} parasitized")
print(f"\ntraining: best epoch {best_epoch['epoch']} of {len(history)} run (max {config.MAX_EPOCHS}), "
      f"val loss {best_epoch['val_loss']:.4f}")

validation = metrics["validation_at_tuned_threshold"]
print(f"\ntuned threshold (chosen on validation): {metrics['tuned_threshold']:.4f} "
      f"-> validation recall {validation['recall']:.4f} (target {metrics['target_recall']})")

test_default = metrics["test_at_default_threshold"]
test_tuned = metrics["test_at_tuned_threshold"]
print(f"\nTEST SET ({metrics['num_test_images']} cells)     default {test_default['threshold']:.4f}"
      f"     tuned {test_tuned['threshold']:.4f}")
for key in ["accuracy", "precision", "recall", "specificity", "f1"]:
    print(f"  {key:<24} {test_default[key]:>10.4f} {test_tuned[key]:>16.4f}")
for key in ["false_negatives", "false_positives", "true_positives", "true_negatives"]:
    print(f"  {key:<24} {test_default[key]:>10} {test_tuned[key]:>16}")
print(f"  {'ROC-AUC':<24} {metrics['test_roc_auc']:>10.4f}")

## Done when

Training finished, the curves look sensible, and the test results are in.

### Explain it back (Phase 4)
1. Looking at *your* curves: did the model overfit? How do you know?
2. Why did training stop when it did?

### Explain it back (Phase 5)
1. Define recall and specificity using "infected cells" and "healthy cells".
2. What did lowering the threshold do to recall and to false alarms?
3. Why must the threshold be chosen on validation data, not test data?